# LFW Grad-CAM — 01. Deterministic case selection

임의의 보기 좋은 이미지를 고르지 않습니다. fallback-free 정량
결과에서 profile별 `stable`, `high_error`, `rank_flip`,
`threshold_crossing` 사례를 결정적으로 선택하고 case manifest를
먼저 고정합니다.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(D:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

MODEL_NAME = "arcface"     # "arcface", "adaface", "magface" 중 이번 실행 모델
MODE = "dev"               # 빠른 검증은 "dev", 전체 논문 실행만 "real"
DATA_FRACTION = 0.10       # identity 단위 사용 비율; 0 < 값 <= 1
SEED = 42                  # 부분집합·tie-break·random control 재현 seed
EXECUTE_STAGE = False      # 입력과 checkpoint를 채운 뒤 실제 계산할 때만 True
WRITE_OUTPUTS = False      # 검증 후 새 artifact를 저장할 때만 True

if MODEL_NAME not in CONFIG["models"]["selected"]:
    raise ValueError(f"지원하지 않는 모델: {MODEL_NAME}")
if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")

In [ ]:
import pandas as pd

from research.explainability.gradcam import select_gradcam_cases

PAIRED_METRICS_PATH = None
RETRIEVAL_METRICS_PATH = None
CASE_MANIFEST_PATH = None
CASES_PER_GROUP = CONFIG["gradcam"]["case_selection"]["samples_per_stratum"]

In [ ]:
if EXECUTE_STAGE:
    if PAIRED_METRICS_PATH is None or RETRIEVAL_METRICS_PATH is None:
        raise RuntimeError("두 정량 결과 경로를 지정하세요.")
    paired = pd.read_parquet(PAIRED_METRICS_PATH)
    retrieval = pd.read_parquet(RETRIEVAL_METRICS_PATH)
    cases = select_gradcam_cases(
        paired,
        retrieval,
        cases_per_group=CASES_PER_GROUP,
        seed=SEED,
    )
    expected_groups = set(CONFIG["gradcam"]["case_selection"]["strata"])
    if not set(cases["case_group"]).issubset(expected_groups):
        raise ValueError("case selector와 설정의 strata가 일치하지 않습니다.")
    if WRITE_OUTPUTS:
        if CASE_MANIFEST_PATH is None:
            raise RuntimeError("CASE_MANIFEST_PATH를 지정하세요.")
        destination = Path(CASE_MANIFEST_PATH).resolve()
        if destination.exists():
            raise FileExistsError(f"기존 case manifest를 덮어쓸 수 없습니다: {destination}")
        destination.parent.mkdir(parents=True, exist_ok=True)
        cases.to_parquet(destination, index=False)
    case_counts = (
        cases.groupby(
            ["compression_family", "compression_profile", "case_group"],
            dropna=False,
        )
        .size()
        .rename("count")
        .reset_index()
    )
else:
    case_counts = {
        "status": "not_executed",
        "reason": "EXECUTE_STAGE=False",
    }
case_counts

이후 노트북은 이 manifest의 `case_id` 순서를 그대로 사용합니다.
사례 수가 부족한 group은 억지로 복제하지 않고 실제 선택 수를 보고합니다.